In [2]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [3]:
import os
import pickle
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
from pymoo.indicators.hv import Hypervolume

from util import natural_key, load_yaml

In [4]:
def load_pareto_history(filepath="pareto_history.pkl"):
    """
    Carga el archivo pickle que contiene fronts_history.
    Devuelve un dict: {generacion: {nivel_frente: [registros...]}}
    """
    with open(filepath, "rb") as f:
        history = pickle.load(f)
    return history

In [5]:
def pareto_history_to_df(filepath="pareto_history.pkl", generation=None):
    """
    Convierte la historia de Pareto guardada en filepath en un DataFrame
    para la generación 'generation', con columnas:
        generation, front_level, accuracy, params, inference_time
    """
    history = load_pareto_history(filepath)
        
    if generation is None:
        generation = max(history.keys())
        
    gen_dict = history[generation]
    records = []
    for level, recs in gen_dict.items():
        # saltamos la métrica hypervolume
        if level == "hypervolume":
            continue
        for rec in recs:
            records.append({
                "id":             rec["id"],
                "generation":      generation,
                "front_level":     level,
                "accuracy":        rec["accuracy"],
                "params":          rec["params"],
                "inference_time":  rec["inference_time"]
            })
    return pd.DataFrame(records)

In [6]:
def archive_to_df(experiment_path: str, archive_subdir: str = "archive") -> pd.DataFrame:
    """Load training metrics from the archive directory into a DataFrame.

    Each folder inside ``archive_subdir`` should contain a ``training_params.txt``
    file with keys such as ``fitness_metric``, ``cuda_inference_time`` and
    ``total_params``. The value for ``fitness_metric`` is taken from the metric
    name specified in that file.

    Parameters
    ----------
    experiment_path : str
        Path to the root experiment directory.
    archive_subdir : str, optional
        Name of the subdirectory that stores archived results. Defaults to
        ``"archive"``.

    Returns
    -------
    pandas.DataFrame
        DataFrame with columns ``id``, ``fitness_metric``, ``cuda_inference_time``
        and ``total_params`` for all archived runs.
    """

    archive_dir = os.path.join(experiment_path, archive_subdir)
    if not os.path.isdir(archive_dir):
        raise FileNotFoundError(f"Archive folder not found at {archive_dir}")

    records = []
    for folder in sorted(os.listdir(archive_dir), key=natural_key):
        params_file = os.path.join(archive_dir, folder, "training_params.txt")
        if not os.path.isfile(params_file):
            continue

        params = load_yaml(params_file)
        metric_key = params.get("fitness_metric")
        if metric_key is None:
            continue

        fitness_value = params.get(metric_key)
        cuda_time = params.get("cuda_inference_time")
        tot_params = params.get("total_params")

        records.append(
            {
                "id": folder,
                "fitness_metric": fitness_value,
                "cuda_inference_time": cuda_time,
                "total_params": tot_params,
            }
        )

    return pd.DataFrame(records)

In [7]:
path_archive = "experiment_cifar10_nsgaX/exp1_repeat_1"
df_archive = archive_to_df(path_archive)
df_archive.shape

(57, 4)

In [8]:
df_archive

,id,fitness_metric,cuda_inference_time,total_params
0,3_5,72.0,1350.355148,928106
1,13_16,68.8,4329.228401,483466
2,13_19,70.0,8633.565903,368394
3,14_2,64.2,3284.907341,502634
4,14_5,71.5,1269.936562,863786
5,14_6,70.6,9605.908394,382826
6,14_12,72.5,3805.470467,613258
7,18_18,74.3,1395.940781,1283082
8,19_1,68.6,22642.707825,242154
9,22_8,66.8,15741.825104,212362


In [9]:
pkl_path = os.path.join(path_archive, "pareto_history.pkl")
df_history = pareto_history_to_df(pkl_path)
df_history = df_history[df_history["front_level"] == 1]

In [12]:
# check if there are some ids that has the same values in accuracy, params and inference_time, in df_history, if so, remove the ones that are not in df_archive
duplicated_rows = df_history[df_history.duplicated(subset=['accuracy', 'params', 'inference_time'], keep=False)]
if not duplicated_rows.empty:
    print("Duplicated rows found")
    archive_ids = set(df_archive['id'])
    rows_to_remove = duplicated_rows[~duplicated_rows['id'].isin(archive_ids)]
    if not rows_to_remove.empty:
        print(f"Removing {len(rows_to_remove)} rows")
        df_history = df_history.drop(rows_to_remove.index)


Duplicated rows found
Removing 41 rows


In [13]:
df_history

,id,generation,front_level,accuracy,params,inference_time
3,13_16,49,1,68.8,483466.0,4329.228401
4,13_19,49,1,70.0,368394.0,8633.565903
5,14_12,49,1,72.5,613258.0,3805.470467
6,14_2,49,1,64.2,502634.0,3284.907341
7,14_5,49,1,71.5,863786.0,1269.936562
8,14_6,49,1,70.6,382826.0,9605.908394
9,18_18,49,1,74.3,1283082.0,1395.940781
10,19_1,49,1,68.6,242154.0,22642.707825
15,22_12,49,1,50.8,52746.0,10984.992981
16,22_14,49,1,58.9,102922.0,15058.159828


In [14]:
# check if there are any different ids in the archive and history
archive_ids = set(df_archive["id"])
history_ids = set(df_history["id"])
print(f"Archive IDs: {len(archive_ids)}")
print(f"History IDs: {len(history_ids)}")
print(f"Different IDs: {len(archive_ids - history_ids)}")
# check the values of the ids that are in the archive but not in the history
diff_ids = archive_ids - history_ids
print("Different IDs:", diff_ids)

Archive IDs: 57
History IDs: 57
Different IDs: 0
Different IDs: set()
